In [ ]:
import pandas as pd
import glob

# Find your file
files = glob.glob("*.csv")
if files:
    # SKIPROWS=5 is the magic fix for your specific error!
    master_df = pd.read_csv(files[0], encoding='cp949', skiprows=5)

    # Check if it worked
    print(f"Success! Found {len(master_df.columns)} columns.")
    display(master_df.head(3))
else:
    print("No file found. Please upload the CSV to the sidebar folder first!")

Success! Found 11 columns.


,2025-11-01,02618,2618. 삼전역 1번출구,정기권,Unnamed: 4,~10대,1,36.30,0.38,1636.82,12
0,2025-11-01,2646,2646.레이크펠리스101동앞,정기권,NaN,~10대,1,46.94,0.42,1823.69,12
1,2025-11-01,2538,2538.사평리근린공원,정기권,NaN,~10대,1,30.78,0.28,1195.62,9
2,2025-11-01,2721,2721.등촌1-10단지 교차로,정기권,NaN,~10대,1,16.40,0.17,753.13,5


In [ ]:
import pandas as pd
import glob

# 1. Find the 2025 Subscriber files
# This looks for files with '25' in the name
files_2025 = glob.glob("*.csv")

if not files_2025:
    print(" No 2025 files found! Please check your sidebar.")
else:
    # 2. Merge while skipping the 5 header rows (the metadata)
    # We use 'cp949' for Korean CSV encoding
    df_list = [pd.read_csv(f, encoding='cp949', skiprows=5) for f in files_2025]
    master_df = pd.concat(df_list, ignore_index=True)

    # Clean up column names
    master_df.columns = master_df.columns.str.strip()

    print(f" Mega-Merge Complete! You now have a full year of 2025 data.")
    print(f"Total rows: {len(master_df)}")
    display(master_df.head(3))

 Mega-Merge Complete! You now have a full year of 2025 data.
Total rows: 5002092


,2025-11-01,02618,2618. 삼전역 1번출구,정기권,Unnamed: 4,~10대,1,36.30,0.38,1636.82,...,12.89,0.14,591.77,3,2025-12-01,03516,3516. 구의아리수정수센터앞,42.08,0.50,2168.45
0,2025-11-01,2646.0,2646.레이크펠리스101동앞,정기권,NaN,~10대,1,46.94,0.42,1823.69,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-11-01,2538.0,2538.사평리근린공원,정기권,NaN,~10대,1,30.78,0.28,1195.62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-11-01,2721.0,2721.등촌1-10단지 교차로,정기권,NaN,~10대,1,16.40,0.17,753.13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Quick analysis of your merged data
print("--- Quick Statistics ---")
if '가입자수' in master_df.columns:
    total_new_users = master_df['가입자수'].sum()
    print(f"Total New Subscribers in 2025: {total_new_users}")
else:
    # If column names are different, show the counts of the 'Age' column
    print(master_df.iloc[:, 2].value_counts()) # Shows Age Group counts

--- Quick Statistics ---
2618. 삼전역 1번출구
2715.마곡나루역 2번 출구             2494
1210. 롯데월드타워(잠실역2번출구 쪽)      2418
2728.마곡나루역 3번 출구             2150
2701. 마곡나루역 5번출구 뒤편          2105
1153. 발산역 1번, 9번 인근 대여소      2050
                             ... 
1278. 송파구청 교차로                  4
1362. 보문역6번출구 앞                 3
1845. 롯데캐슬골드파크1차 서문             3
3605. 봉은사역6번출구(현대아이파크타워앞)       1
211. 여의도역 4번출구 옆                1
Name: count, Length: 2773, dtype: int64


In [ ]:
import pandas as pd
import gradio as gr

# 1. Load and Merge Data
def load_and_merge():
    # Note: Ensure these files are in your directory
    file_list = [
        '서울특별시 공공자전거 이용정보(일별)_2510.csv',
        '서울특별시 공공자전거 이용정보(일별)_2511.csv',
        '서울특별시 공공자전거 이용정보(일별)_2512.csv'
    ]

    list_df = []
    for file in file_list:
        try:
            # Using 'cp949' encoding for Korean text files
            temp_df = pd.read_csv(file, encoding='cp949')
            list_df.append(temp_df)
            print(f"Successfully loaded: {file}")
        except Exception as e:
            print(f"Error loading {file}: {e}")

    if not list_df:
        return pd.DataFrame()

    full_df = pd.concat(list_df, ignore_index=True)
    return full_df

# Pre-load the data
df_bike = load_and_merge()

# 2. Analysis Functions
def get_summary():
    # Returns basic statistics (mean, count, etc.)
    return df_bike.describe()

def get_top_stations(n=10):
    # '대여소명' is the Korean column for 'Station Name'
    col_name = '대여소명' if '대여소명' in df_bike.columns else df_bike.columns[1]

    # Process top stations and rename columns for the English UI
    result = df_bike[col_name].value_counts().head(n).reset_index()
    result.columns = ['Station Name', 'Usage Count']
    return result

# 3. Create Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# 🚲 Seoul Public Bike Usage Dashboard (Q4 2025)")

    with gr.Tabs():
        # Tab 1: Raw Data
        with gr.TabItem("Raw Data View"):
            gr.Markdown("### Sample of the top 100 rows")
            gr.DataFrame(df_bike.head(100), label="Data Sample")

        # Tab 2: Statistics
        with gr.TabItem("Statistical Summary"):
            btn_stat = gr.Button("Generate Statistics")
            stat_output = gr.DataFrame(label="Basic Statistical Analysis")
            btn_stat.click(get_summary, outputs=stat_output)

        # Tab 3: Station Rankings
        with gr.TabItem("Popular Stations"):
            gr.Markdown("### Top 10 Most Used Rental Stations")
            btn_rank = gr.Button("Show Top Stations")
            rank_output = gr.DataFrame(label="Station Rankings")
            btn_rank.click(get_top_stations, outputs=rank_output)

# Launch the app
if __name__ == "__main__":
    demo.launch()

Successfully loaded: 서울특별시 공공자전거 이용정보(일별)_2510.csv
Successfully loaded: 서울특별시 공공자전거 이용정보(일별)_2511.csv
Successfully loaded: 서울특별시 공공자전거 이용정보(일별)_2512.csv
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad78a5034c413d44ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import pandas as pd
import gradio as gr
import google.generativeai as genai
import matplotlib.pyplot as plt
import os

# --- CONFIGURATION ---
# On Hugging Face, set this in Settings > Secrets
api_key = os.getenv("GOOGLE_API_KEY", "YOUR_KEY_HERE")
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-1.5-flash')

# Load data once at startup for speed
try:
    # Use a sample or the actual file path
    master_df = pd.read_csv('seoul_bike_data.csv', encoding='cp949').fillna(0)
except:
    # Fallback empty dataframe for testing
    master_df = pd.DataFrame(columns=['대여소명', '이용거리', '이용시간'])

def quick_analyze(row_num, lang):
    try:
        row = master_df.iloc[int(row_num)]
        station = row.get('대여소명', 'Unknown')
        dist = row.get('이용거리', 0)
        time = row.get('이용시간', 0)

        # AI Prompt - Force short response for speed
        prompt = f"Explain this Seoul bike trip in {lang} (2 sentences max): Station {station}, {dist}m, {time}min."
        response = model.generate_content(prompt)

        # Fast Plotting
        fig, ax = plt.subplots(figsize=(4, 2))
        ax.barh(['Dist (m)', 'Time (min)'], [float(dist), float(time)], color=['#34d399', '#3b82f6'])
        plt.tight_layout()

        return response.text, fig

    except Exception as e:
        return f"Error: {str(e)}", None

# --- GRADIO UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🚲 Seoul Bike Fast AI")

    with gr.Row():
        with gr.Column(scale=1):
            idx = gr.Number(label="Row ID", value=0, precision=0)
            lang = gr.Dropdown(["English", "Korean", "Filipino"], label="Lang", value="English")
            btn = gr.Button("Analyze ⚡", variant="primary")

        with gr.Column(scale=2):
            out_txt = gr.Markdown("Analysis will appear here...")
            out_plot = gr.Plot(label="Trip Stats")

    btn.click(quick_analyze, [idx, lang], [out_txt, out_plot])

demo.launch()

if row_num >= len(master_df):
    return f"Error: Please enter a Row ID between 0 and {len(master_df)-1}", None

In [ ]:
from google.colab import userdata
userdata.get('cn')

'cn'

In [ ]:
from google.colab import userdata
import os
from google import genai
from google.genai import types

# Using 'cn' as requested
GEMINI_API_KEY = userdata.get('cn')
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

client = genai.Client(api_key=GEMINI_API_KEY)

# Selecting the model
MODEL_ID = "gemini-3.1-pro-preview"

In [ ]:
import gradio as gr

def generate_text_with_gemini(prompt):
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"An error occurred: {e}"

# Build the Interface
iface = gr.Interface(
    fn=generate_text_with_gemini,
    inputs=gr.Textbox(lines=5, label="Enter your text prompt:"),
    outputs=gr.Textbox(label="Generated Text:"),
    title="Gemini Text Generation App (CN)",
    description="Custom text generation powered by Gemini."
)

# Launch the app
iface.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://38b869106cc6f6e2ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
